# 🔬 Agricultural Pest Image Retrieval System (CBIR) using DINOv2 / ViT on Kaggle
Notebook này hướng dẫn chi tiết cách thiết lập môi trường, chạy hệ thống truy vấn ảnh sâu bệnh nông nghiệp 2 giai đoạn (**Automatic Detection & Crop -> DinoV2/ViT Search**) trên tập dữ liệu **IP102** trên Kaggle cho cả **4 Tasks** nhằm thu thập chỉ số Recall@1/5/10 cho báo cáo đánh giá.

### ⚠️ Yêu cầu trước khi chạy:
1. Chắc chắn rằng bạn đã kích hoạt **GPU T4 x2** hoặc **GPU P100** trong phần settings của Kaggle (*Accelerator -> GPU*).
2. Bật kết nối internet cho notebook (*Internet on*). Nếu chạy offline hoàn toàn, vui lòng chuẩn bị sẵn dataset chứa weights của mô hình `facebook/dinov2-base`.

## 🛠️ Bước 1: Thiết lập môi trường và tải repository

In [1]:
# 1. Clone repository chứa code mới nhất
import os
repo_url = "https://github.com/nta2112/OW_OVD-An-custom.git"
working_dir = "/kaggle/working/OW_OVD"

if not os.path.exists(working_dir):
    print("-> Đang clone repository từ GitHub...")
    !git clone {repo_url} {working_dir}
else:
    print("-> Repository đã tồn tại. Đang tiến hành cập nhật (git pull)...")
    %cd {working_dir}
    !git pull

%cd {working_dir}

# 2. Tải submodule mmyolo
if not os.path.exists("third_party/mmyolo"):
    print("-> Đang tải submodule mmyolo...")
    !git clone https://github.com/open-mmlab/mmyolo.git third_party/mmyolo
else:
    print("-> Submodule mmyolo đã có sẵn.")

-> Đang clone repository từ GitHub...
Cloning into '/kaggle/working/OW_OVD'...
remote: Enumerating objects: 1415, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 1415 (delta 6), reused 10 (delta 2), pack-reused 1391 (from 1)
Receiving objects: 100% (1415/1415), 2.70 MiB | 26.58 MiB/s, done.
Resolving deltas: 100% (970/970), done.
/kaggle/working/OW_OVD
-> Đang tải submodule mmyolo...
Cloning into 'third_party/mmyolo'...
remote: Enumerating objects: 4968, done.
remote: Counting objects: 100% (1341/1341), done.
remote: Compressing objects: 100% (294/294), done.
remote: Total 4968 (delta 1133), reused 1047 (delta 1047), pack-reused 3627 (from 1)
Receiving objects: 100% (4968/4968), 3.62 MiB | 12.87 MiB/s, done.
Resolving deltas: 100% (3216/3216), done.


## 📦 Bước 2: Cài đặt Dependencies & Vá lỗi phiên bản MMCV

In [2]:
print("-> 1. Cài đặt các gói PyTorch & Torchvision tương thích cu121...")
!pip install -q torch==2.4.0+cu121 torchvision==0.19.0+cu121 --extra-index-url https://download.pytorch.org/whl/cu121

print("\n-> 2. Cài đặt MMCV từ pre-built index...")
!pip install -q mmcv -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.4/index.html

print("\n-> 3. Cài đặt các thư viện bổ trợ...")
!pip install -q matplotlib pycocotools terminaltables mmengine prettytable wcwidth open_clip_torch transformers

print("\n-> 4. Cài đặt MMDetection...")
!pip install -q "mmdet>=3.1.0" --no-deps

print("\n-> 5. Cài đặt MMYOLO từ source...")
!pip install -q --no-build-isolation --no-deps third_party/mmyolo

print("\n-> 6. Vá lỗi kiểm tra phiên bản MMCV...")
import site
import glob

site_dirs = site.getsitepackages()
for s_dir in site_dirs:
    for pkg in ["mmdet", "mmyolo"]:
        init_file = os.path.join(s_dir, pkg, "__init__.py")
        if os.path.exists(init_file):
            with open(init_file, 'r', encoding='utf-8') as f:
                content = f.read()
            content = content.replace("mmcv_maximum_version = '2.2.0'", "mmcv_maximum_version = '2.3.0'")
            content = content.replace("mmcv_maximum_version = '2.1.0'", "mmcv_maximum_version = '2.3.0'")
            with open(init_file, 'w', encoding='utf-8') as f:
                f.write(content)

print("====== Khởi tạo môi trường thành công! ======")

-> 1. Cài đặt các gói PyTorch & Torchvision tương thích cu121...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.0/799.0 MB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 109.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 208.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 217.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 130.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 70.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 139.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 173.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 148.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 120.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17

## 🗂️ Bước 3: Tìm kiếm Dataset & Định nghĩa Checkpoints

In [3]:
import glob

# Tự động định vị thư mục dataset IP102 trên Kaggle
dataset_root = None
for path in [
    '/kaggle/input/datasets/nta212/ip102-for-object-detection',
    '/kaggle/input/ip102-for-object-detection',
    'data/IP102',
    '.'
]:
    if os.path.exists(os.path.join(path, 'train.json')):
        dataset_root = path
        break

if dataset_root is None:
    paths = glob.glob('/kaggle/input/datasets/nta212/ip102-for-object-detection/**/train.json', recursive=True)
    if paths:
        dataset_root = os.path.dirname(paths[0])

print(f"-> Thư mục Dataset IP102: {dataset_root}")

# ĐƯỜNG DẪN CHECKPOINT CỦA BẠN TRÊN KAGGLE
CHECKPOINT_DIR = "/kaggle/input/models/nta212/task-1-ow-ovd-25-class/pytorch/default/3"

# Xác nhận sự tồn tại của file checkpoint mẫu của Task 1 để kiểm tra
t1_ckpt = os.path.join(CHECKPOINT_DIR, "best_coco_Current class AP50_epoch_5.pth")
if os.path.exists(t1_ckpt):
    print(f"-> Phát hiện checkpoint Task 1 tại: {t1_ckpt}")
else:
    print(f"⚠️ Cảnh báo: Chưa tìm thấy checkpoint tại '{t1_ckpt}'. Vui lòng cập nhật biến CHECKPOINT_DIR.")

-> Thư mục Dataset IP102: /kaggle/input/datasets/nta212/ip102-for-object-detection
-> Phát hiện checkpoint Task 1 tại: /kaggle/input/models/nta212/task-1-ow-ovd-25-class/pytorch/default/3/best_coco_Current class AP50_epoch_5.pth


## 🛠️ Bước 3b: Sinh các tệp đặc trưng & thuộc tính phụ trợ (Auxiliary Embeddings)
Mô hình YOLO-World/OW-OVD yêu cầu các file vector đặc trưng lớp gán nhãn, thuộc tính, và phân phối mẫu để khởi tạo Box Head. Cell này tự động sinh các file này trong thư mục `data/IP102` trước khi nạp mô hình.

In [4]:
import json
import torch
import numpy as np
import os
from transformers import AutoTokenizer, CLIPTextModelWithProjection

# 1. Khởi tạo các thư mục con
os.makedirs('pretrained_models', exist_ok=True)
os.makedirs('data/IP102', exist_ok=True)
os.makedirs('data/texts/IP102', exist_ok=True)

# 2. Tải pretrain weights gốc của YOLO-World làm nền tảng nếu cần
weights_path = 'pretrained_models/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth'
if not os.path.exists(weights_path):
    print("-> Đang tải pretrained weights gốc...")
    !wget -O {weights_path} https://huggingface.co/wondervictor/YOLO-World/resolve/main/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth
else:
    print("-> Pretrained weights gốc đã có sẵn.")

# 3. Đọc tên các class từ file annotations IP102
ann_path = os.path.join(dataset_root, 'train.json')
if os.path.exists(ann_path):
    with open(ann_path, 'r') as f:
        coco_data = json.load(f)
    categories = sorted(coco_data['categories'], key=lambda x: x['id'])
    class_names = [cat['name'] for cat in categories]
else:
    print("-> Không tìm thấy train.json. Sử dụng danh sách class fallback...")
    class_names = ['14', '15', '16', '18', '22', '23', '24', '25', '26', '37', '38', '39', '45', '46', '47', '48', '49', '50', '51', '66', '67', '69', '70', '86', '101']

num_classes = len(class_names)
print(f"-> Tổng số lớp học (classes): {num_classes}")

# 4. Lưu file class_texts.json
class_texts = [[name] for name in class_names]
with open('data/texts/IP102/class_texts.json', 'w') as f:
    json.dump(class_texts, f)

# 5. Sinh class embeddings bằng CLIP
print("-> Đang trích xuất text embeddings bằng CLIP...")
model_name = 'openai/clip-vit-base-patch32'
tokenizer = AutoTokenizer.from_pretrained(model_name)
clip_model = CLIPTextModelWithProjection.from_pretrained(model_name, use_safetensors=True)
clip_model.eval()

embeddings = []
with torch.no_grad():
    for name in class_names:
        inputs = tokenizer(name, padding=True, return_tensors="pt")
        outputs = clip_model(**inputs)
        embed = outputs.text_embeds[0].cpu().numpy()
        embed = embed / np.linalg.norm(embed)
        embeddings.append(embed)

np.save('data/IP102/ip102_gt_embeddings.npy', np.array(embeddings))
print("-> Đã lưu ip102_gt_embeddings.npy")

# 6. Sinh file task_att_1_embeddings.pth
num_att = num_classes * 25
torch.save({
    'att_embedding': torch.zeros(num_att, 512),
    'att_text': [f"att_{i}" for i in range(num_att)]
}, 'data/IP102/task_att_1_embeddings.pth')
print("-> Đã lưu task_att_1_embeddings.pth")

# 7. Sinh file mowod_distribution_sim1.pth
thrs = [0.55]
pos_dist = [{att_i: torch.zeros(10000) for att_i in range(num_att)} for _ in thrs]
neg_dist = [{att_i: torch.zeros(10000) for att_i in range(num_att)} for _ in thrs]
torch.save({
    'positive_distributions': pos_dist,
    'negative_distributions': neg_dist
}, 'data/IP102/mowod_distribution_sim1.pth')
print("====== Sinh toàn bộ các file đặc trưng phụ trợ thành công! ======")

-> Đang tải pretrained weights gốc...
--2026-08-28 02:02:16--  https://huggingface.co/wondervictor/YOLO-World/resolve/main/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth
Resolving huggingface.co (huggingface.co)... 52.85.193.32, 52.85.193.24, 52.85.193.123, ...
Connecting to huggingface.co (huggingface.co)|52.85.193.32|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/65bb7a71626a4c209906adf5/09dafb73b0d19d270cf20f7eeac6a7861303a753332d5df9917772ba23e4a47d?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth%3B+filename%3D%22yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth%22%3B&user_id=public&X-Xet-Cas-Uid=public&Expires=1787886136&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjViYjdhNzE2MjZhNGMyMDk5MDZhZGY1LzA5ZGFmYjczYjBkMTlkMjcwY2YyMGY3ZWVhYzZhNzg2MTMwM2E3NTMzMzJkNWRmOTkxNzc3MmJhMjNlNGE0N2RcXD9y

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

CLIPTextModelWithProjection LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
vision_model.embeddings.class_embedding                        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
vision_model.encoder.la

-> Đã lưu ip102_gt_embeddings.npy
-> Đã lưu task_att_1_embeddings.pth
====== Sinh toàn bộ các file đặc trưng phụ trợ thành công! ======


## 🔍 Bước 4: Chạy thử tìm kiếm tương đồng trên một ảnh bằng DINOv2 / ViT
Cell này thực hiện quy trình tìm kiếm ảnh sâu bệnh tương đồng sử dụng mô hình trích xuất đặc trưng **DINOv2** (`facebook/dinov2-base`). Bạn có thể đổi sang ViT bất kỳ bằng cách đổi giá trị các biến `extractor_type` và `extractor_model`.

In [5]:
from IPython.display import Image, display
import glob

# --- CẤU HÌNH KIỂU TRÍCH XUẤT ĐẶC TRƯNG ---
# extractor_type = "dinov2"                 # Lựa chọn: clip, dinov2, vit
# extractor_model = "facebook/dinov2-base"  # Repository HF hoặc đường dẫn local
extractor_type = "vit"
extractor_model = "/kaggle/input/models/oleksandrkharytonov/googlevit-base-patch16-224/pytorch/default/1"
# --------------------------------------------

t1_config = "configs/open_world/mowod/custom/ip102_t1.py"
t1_checkpoint = os.path.join(CHECKPOINT_DIR, "best_coco_Current class AP50_epoch_5.pth")
gallery_folder = os.path.join(dataset_root, "test")
ann_file = os.path.join(dataset_root, "test.json")

# Tự động tìm kiếm ảnh thực tế bất kỳ trong tập test để truy vấn tránh FileNotFoundError
test_images = glob.glob(os.path.join(dataset_root, "**/JPEGImages/*.jpg"), recursive=True)
if not test_images:
    test_images = glob.glob(os.path.join(dataset_root, "**/*.jpg"), recursive=True)
if test_images:
    query_image_path = test_images[0]
    print(f"-> Tự động tìm thấy ảnh query thực tế: {query_image_path}")
else:
    query_image_path = os.path.join(dataset_root, "test", "00001.jpg")
    print(f"-> Cảnh báo: Không tìm thấy ảnh .jpg. Dùng fallback: {query_image_path}")

# 1. Xây dựng index cho Gallery bằng extractor đã chọn
print(f"-> Đang khởi tạo CSDL đặc trưng ảnh mẫu (Gallery Index) sử dụng {extractor_type.upper()}... (Có thể mất 2-3 phút)")
!python build_gallery_index.py \
    --config "{t1_config}" \
    --checkpoint "{t1_checkpoint}" \
    --gallery-dir "{gallery_folder}" \
    --ann-file "{ann_file}" \
    --extractor-type {extractor_type} \
    --extractor-model {extractor_model} \
    --output gallery_index_task1.pkl

# 2. Tiến hành truy vấn ảnh
print("\n-> Đang chạy truy vấn ảnh sâu bệnh...")
!python retrieve.py \
    --config "{t1_config}" \
    --checkpoint "{t1_checkpoint}" \
    --gallery-index gallery_index_task1.pkl \
    --query-image "{query_image_path}" \
    --extractor-type {extractor_type} \
    --extractor-model {extractor_model} \
    --top-k 5 \
    --anomaly-thr 0.55 \
    --output query_result.jpg

# 3. Hiển thị kết quả tìm kiếm trực quan trực tiếp trong notebook
if os.path.exists("query_result.jpg"):
    display(Image(filename="query_result.jpg"))
else:
    print("Error: Không tìm thấy ảnh kết quả 'query_result.jpg'.")

-> Cảnh báo: Không tìm thấy ảnh .jpg. Dùng fallback: /kaggle/input/datasets/nta212/ip102-for-object-detection/test/00001.jpg
-> Đang khởi tạo CSDL đặc trưng ảnh mẫu (Gallery Index) sử dụng VIT... (Có thể mất 2-3 phút)
/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \
      IP102 CBIR GALLERY INDEX BUILDER (OFFLINE)      
-> Loaded annotations mapping for 2713 images.
-> WARNING: Specified directory empty. Scanning parent /kaggle/input/datasets/nta212/ip102-for-object-detection recursively...
-> Filtering gallery images using split annotations: /kaggle/input/datasets/nta212/ip102-for-object-detection/test.json
-> Selected 2713 images belonging to split defined in /kaggle/input/datasets/nta212/ip102-for-object-detection/test.json


## 📈 Bước 5: Chạy đánh giá đo chỉ số Recall@1/5/10 cho cả 4 Tasks sử dụng DINOv2 / ViT
Cell này chạy đánh giá tuần tự cho cả 4 task bằng mô hình trích xuất đặc trưng được chỉ định và xuất báo cáo điểm số chi tiết từng loài sâu bệnh ra các file markdown.

In [6]:
# --- CẤU HÌNH ĐÁNH GIÁ (GIỐNG BƯỚC 4) ---
# extractor_type = "dinov2"
# extractor_model = "facebook/dinov2-base"
extractor_type = "vit"
extractor_model = "/kaggle/input/models/oleksandrkharytonov/googlevit-base-patch16-224/pytorch/default/1"
# ------------------------------------------

tasks = [
    {"id": 1, "config": "configs/open_world/mowod/custom/ip102_t1.py", "checkpoint": os.path.join(CHECKPOINT_DIR, "best_coco_Current class AP50_epoch_5.pth")},
    {"id": 2, "config": "configs/open_world/mowod/custom/ip102_t2.py", "checkpoint": os.path.join(CHECKPOINT_DIR, "ip102_t2.pth")},
    {"id": 3, "config": "configs/open_world/mowod/custom/ip102_t3.py", "checkpoint": os.path.join(CHECKPOINT_DIR, "ip102_t3.pth")},
    {"id": 4, "config": "configs/open_world/mowod/custom/ip102_t4.py", "checkpoint": os.path.join(CHECKPOINT_DIR, "ip102_t4.pth")}
]

for t in tasks:
    task_id = t["id"]
    config_path = t["config"]
    ckpt_path = t["checkpoint"]
    
    print("\n" + "="*60)
    print(f"   ĐANG CHẠY ĐÁNH GIÁ THỰC NGHIỆM CHO TASK {task_id}   ")
    print("="*60)
    
    if not os.path.exists(ckpt_path):
        print(f"⚠️ Bỏ qua Task {task_id} vì không tìm thấy file checkpoint tại: {ckpt_path}")
        continue
        
    # Khởi chạy script đánh giá với extractor được cấu hình
    !python evaluate_retrieval.py \
        --config "{config_path}" \
        --checkpoint "{ckpt_path}" \
        --dataset-root "{dataset_root}" \
        --query-split val \
        --gallery-split test \
        --query-cache "query_cache_task{task_id}_{extractor_type}.pkl" \
        --gallery-cache "gallery_cache_task{task_id}_{extractor_type}.pkl" \
        --extractor-type {extractor_type} \
        --extractor-model {extractor_model} \
        --output-report "report_task{task_id}_{extractor_type}.md"


   ĐANG CHẠY ĐÁNH GIÁ THỰC NGHIỆM CHO TASK 1   
/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \
-> Detected offline Kaggle CLIP model. Defaulting to: /kaggle/input/models/yujkaggle/openaiclip-vit-base-patch32/pytorch/default/1
      IP102 RETRIEVAL METRICS EVALUATION PIPELINE      
-> Reading annotations...
-> WARNING: Standard subfolders empty. Scanning /kaggle/input/datasets/nta212/ip102-for-object-detection recursively to locate image files...
-> Auto-resolved image directory to: /kaggle/input/datasets/nta212/ip102-for-object-detection/VOC2007/VOC2007/JPEGImages
-> WARNING: Standard subfolders empty. Scanning /kaggle/input/datasets/nta212/ip102-for-object-detection recursively to locate image files...
-> Auto-resolved imag

## 📄 Bước 6: Đọc kết quả các Task phục vụ báo cáo

In [7]:
from IPython.display import Markdown, display

for i in [1, 2, 3, 4]:
    report_file = f"report_task{i}_{extractor_type}.md"
    if os.path.exists(report_file):
        print(f"\n\n🔍 KẾT QUẢ ĐÁNH GIÁ CHỈ SỐ TASK {i} (Được đọc từ {report_file}):")
        display(Markdown(filename=report_file))
    else:
        print(f"-> Không tìm thấy báo cáo kết quả của Task {i} (Chưa chạy đánh giá hoặc lỗi file).")



🔍 KẾT QUẢ ĐÁNH GIÁ CHỈ SỐ TASK 1 (Được đọc từ report_task1_vit.md):


# Agricultural Pest Retrieval System Evaluation Report

- **Query Split:** `val`
- **Gallery Split:** `test`

## Summary Metrics

| Metric | Macro Average | Weighted Average |
| :--- | :---: | :---: |
| **Recall@1** | 0.6600 | - |
| **Recall@5** | 0.8753 | - |
| **Recall@10** | 0.9288 | - |

## Open-World Anomaly Detection Metrics (HAUF Safeguard)

| Metric | Score |
| :--- | :---: |
| **AUROC (Area Under ROC)** | 0.4703 |
| **FPR@TPR95** | 0.9577 |

### ROC Curve Plot

![ROC Curve](roc_curve_ip102_t1.png)

## Class-Wise Retrieval Metrics

| Class Name | Query Count | Recall@1 | Recall@5 | Recall@10 |
| :--- | :---: | :---: | :---: | :---: |
| 101 | 470 | 0.9149 | 0.9851 | 0.9936 |
| 14 | 70 | 0.9714 | 0.9857 | 0.9857 |
| 15 | 139 | 0.9784 | 0.9856 | 0.9928 |
| 16 | 68 | 0.8971 | 0.9706 | 0.9706 |
| 18 | 48 | 0.7292 | 0.9375 | 0.9583 |
| 22 | 68 | 0.8088 | 0.9265 | 0.9559 |
| 23 | 33 | 0.4545 | 0.7879 | 0.9091 |
| 24 | 141 | 0.8652 | 0.9645 | 0.9858 |
| 25 | 33 | 0.9091 | 0.9091 | 0.9697 |
| 26 | 39 | 0.8205 | 0.9231 | 0.9231 |
| 37 | 56 | 0.9464 | 1.0000 | 1.0000 |
| 38 | 40 | 0.3000 | 0.6250 | 0.7750 |
| 39 | 66 | 0.3788 | 0.7879 | 0.9091 |
| 45 | 66 | 0.3485 | 0.7576 | 0.8485 |
| 46 | 37 | 0.1892 | 0.5676 | 0.6757 |
| 47 | 56 | 0.3750 | 0.8214 | 0.8929 |
| 48 | 86 | 0.5116 | 0.7791 | 0.9186 |
| 49 | 39 | 0.2821 | 0.6410 | 0.8205 |
| 50 | 70 | 0.5857 | 0.9000 | 0.9429 |
| 51 | 150 | 0.6533 | 0.8867 | 0.9600 |
| 66 | 45 | 0.8889 | 0.9778 | 0.9778 |
| 67 | 54 | 0.9259 | 0.9444 | 0.9444 |
| 69 | 33 | 0.4848 | 0.9394 | 0.9697 |
| 70 | 203 | 0.7192 | 0.9704 | 0.9852 |
| 86 | 66 | 0.5606 | 0.9091 | 0.9545 |




🔍 KẾT QUẢ ĐÁNH GIÁ CHỈ SỐ TASK 2 (Được đọc từ report_task2_vit.md):


# Agricultural Pest Retrieval System Evaluation Report

- **Query Split:** `val`
- **Gallery Split:** `test`

## Summary Metrics

| Metric | Macro Average | Weighted Average |
| :--- | :---: | :---: |
| **Recall@1** | 0.6507 | - |
| **Recall@5** | 0.8698 | - |
| **Recall@10** | 0.9237 | - |

## Open-World Anomaly Detection Metrics (HAUF Safeguard)

| Metric | Score |
| :--- | :---: |
| **AUROC (Area Under ROC)** | 0.4929 |
| **FPR@TPR95** | 0.9556 |

### ROC Curve Plot

![ROC Curve](roc_curve_ip102_t2.png)

## Class-Wise Retrieval Metrics

| Class Name | Query Count | Recall@1 | Recall@5 | Recall@10 |
| :--- | :---: | :---: | :---: | :---: |
| 101 | 470 | 0.9255 | 0.9809 | 0.9957 |
| 14 | 70 | 0.9571 | 0.9857 | 0.9857 |
| 15 | 139 | 0.9928 | 0.9928 | 0.9928 |
| 16 | 68 | 0.9118 | 0.9853 | 1.0000 |
| 18 | 48 | 0.6667 | 0.9583 | 0.9792 |
| 22 | 68 | 0.8088 | 0.9118 | 0.9559 |
| 23 | 33 | 0.4242 | 0.8182 | 0.8788 |
| 24 | 141 | 0.8794 | 0.9645 | 0.9787 |
| 25 | 33 | 0.9091 | 0.9394 | 0.9394 |
| 26 | 39 | 0.8205 | 0.9487 | 0.9487 |
| 37 | 56 | 1.0000 | 1.0000 | 1.0000 |
| 38 | 40 | 0.2250 | 0.6250 | 0.7750 |
| 39 | 66 | 0.3788 | 0.7424 | 0.8636 |
| 45 | 66 | 0.3333 | 0.8030 | 0.9091 |
| 46 | 37 | 0.1892 | 0.5405 | 0.7027 |
| 47 | 56 | 0.2857 | 0.7143 | 0.8750 |
| 48 | 86 | 0.5465 | 0.8023 | 0.8837 |
| 49 | 39 | 0.2821 | 0.5641 | 0.7436 |
| 50 | 70 | 0.5429 | 0.9000 | 0.9000 |
| 51 | 150 | 0.5667 | 0.9133 | 0.9733 |
| 66 | 45 | 0.9111 | 0.9778 | 0.9778 |
| 67 | 54 | 0.9259 | 0.9444 | 0.9444 |
| 69 | 33 | 0.5758 | 0.8788 | 0.9394 |
| 70 | 203 | 0.7094 | 0.9606 | 0.9803 |
| 86 | 66 | 0.5000 | 0.8939 | 0.9697 |




🔍 KẾT QUẢ ĐÁNH GIÁ CHỈ SỐ TASK 3 (Được đọc từ report_task3_vit.md):


# Agricultural Pest Retrieval System Evaluation Report

- **Query Split:** `val`
- **Gallery Split:** `test`

## Summary Metrics

| Metric | Macro Average | Weighted Average |
| :--- | :---: | :---: |
| **Recall@1** | 0.6632 | - |
| **Recall@5** | 0.8649 | - |
| **Recall@10** | 0.9229 | - |

## Open-World Anomaly Detection Metrics (HAUF Safeguard)

| Metric | Score |
| :--- | :---: |
| **AUROC (Area Under ROC)** | 0.5058 |
| **FPR@TPR95** | 0.9616 |

### ROC Curve Plot

![ROC Curve](roc_curve_ip102_t3.png)

## Class-Wise Retrieval Metrics

| Class Name | Query Count | Recall@1 | Recall@5 | Recall@10 |
| :--- | :---: | :---: | :---: | :---: |
| 101 | 470 | 0.9021 | 0.9872 | 0.9936 |
| 14 | 70 | 0.9714 | 0.9714 | 0.9857 |
| 15 | 139 | 0.9928 | 0.9928 | 0.9928 |
| 16 | 68 | 0.8824 | 0.9706 | 0.9853 |
| 18 | 48 | 0.7708 | 0.9792 | 0.9792 |
| 22 | 68 | 0.8088 | 0.9118 | 0.9559 |
| 23 | 33 | 0.5758 | 0.7879 | 0.9394 |
| 24 | 141 | 0.8511 | 0.9504 | 0.9787 |
| 25 | 33 | 0.9091 | 0.9697 | 0.9697 |
| 26 | 39 | 0.7949 | 0.9231 | 0.9487 |
| 37 | 56 | 0.9821 | 1.0000 | 1.0000 |
| 38 | 40 | 0.2000 | 0.5250 | 0.7000 |
| 39 | 66 | 0.3333 | 0.6970 | 0.8636 |
| 45 | 66 | 0.3636 | 0.8030 | 0.8636 |
| 46 | 37 | 0.1892 | 0.4595 | 0.6757 |
| 47 | 56 | 0.3750 | 0.7857 | 0.8929 |
| 48 | 86 | 0.5233 | 0.7791 | 0.8837 |
| 49 | 39 | 0.3077 | 0.5897 | 0.6923 |
| 50 | 70 | 0.5571 | 0.8857 | 0.9429 |
| 51 | 150 | 0.6467 | 0.9133 | 0.9667 |
| 66 | 45 | 0.8667 | 0.9778 | 0.9778 |
| 67 | 54 | 0.9259 | 0.9444 | 0.9444 |
| 69 | 33 | 0.4848 | 0.9394 | 0.9697 |
| 70 | 203 | 0.7143 | 0.9557 | 0.9852 |
| 86 | 66 | 0.6515 | 0.9242 | 0.9848 |




🔍 KẾT QUẢ ĐÁNH GIÁ CHỈ SỐ TASK 4 (Được đọc từ report_task4_vit.md):


# Agricultural Pest Retrieval System Evaluation Report

- **Query Split:** `val`
- **Gallery Split:** `test`

## Summary Metrics

| Metric | Macro Average | Weighted Average |
| :--- | :---: | :---: |
| **Recall@1** | 0.6512 | - |
| **Recall@5** | 0.8589 | - |
| **Recall@10** | 0.9239 | - |

## Class-Wise Retrieval Metrics

| Class Name | Query Count | Recall@1 | Recall@5 | Recall@10 |
| :--- | :---: | :---: | :---: | :---: |
| 101 | 470 | 0.9234 | 0.9851 | 0.9936 |
| 14 | 70 | 0.9714 | 0.9857 | 0.9857 |
| 15 | 139 | 0.9856 | 0.9928 | 0.9928 |
| 16 | 68 | 0.8529 | 0.9559 | 1.0000 |
| 18 | 48 | 0.6667 | 0.9375 | 0.9792 |
| 22 | 68 | 0.8235 | 0.9412 | 0.9559 |
| 23 | 33 | 0.4242 | 0.7273 | 0.8485 |
| 24 | 141 | 0.8369 | 0.9645 | 0.9716 |
| 25 | 33 | 0.9091 | 0.9091 | 0.9697 |
| 26 | 39 | 0.8205 | 0.8974 | 0.9231 |
| 37 | 56 | 0.9643 | 1.0000 | 1.0000 |
| 38 | 40 | 0.2500 | 0.6000 | 0.7500 |
| 39 | 66 | 0.3333 | 0.6818 | 0.8485 |
| 45 | 66 | 0.3636 | 0.7121 | 0.8788 |
| 46 | 37 | 0.2162 | 0.5135 | 0.7297 |
| 47 | 56 | 0.3750 | 0.7679 | 0.8750 |
| 48 | 86 | 0.5116 | 0.7791 | 0.9070 |
| 49 | 39 | 0.2821 | 0.6154 | 0.7436 |
| 50 | 70 | 0.5000 | 0.8857 | 0.9286 |
| 51 | 150 | 0.6667 | 0.9467 | 0.9667 |
| 66 | 45 | 0.8667 | 0.9778 | 1.0000 |
| 67 | 54 | 0.9259 | 0.9259 | 0.9444 |
| 69 | 33 | 0.5758 | 0.9091 | 0.9697 |
| 70 | 203 | 0.7340 | 0.9507 | 0.9803 |
| 86 | 66 | 0.5000 | 0.9091 | 0.9545 |
